%md
<div style="display: flex; align-items: center; gap: 18px; margin-bottom: 15px;">
  <img src="https://files.codebasics.io/v3/images/sticky-logo.svg" alt="Codebasics Logo" style="display: inline-block;" width="130">
  <h1 style="font-size: 34px; color: #1f4e79; margin: 0; display: inline-block;">Codebasics Practice Room - Data Engineering Bootcamp </h1>
</div>

## Setup

In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType
from delta.tables import DeltaTable

In [0]:
# Databricks Unity Catalog Setup

from pyspark.sql import SparkSession

catalog_name = "practice_db_catalog"
schema_name = "clickstream"
volume_name = "data_volume"
folder_name = "clickstream"

# Create Catalog if not exists 

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
print(f"Catalog '{catalog_name}' ready.")

# create Schema (database) if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
print(f"Schema '{schema_name}' created inside '{catalog_name}'.")

# create volume if not exists

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
print(f"Volume '{volume_name}' created inside '{catalog_name}.{schema_name}'.")

# Set current context 
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {schema_name}")

# Define volume backed paths 
base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
raw_path = f"{base_path}/raw"

# create directories inside the volume 
dbutils.fs.mkdirs(raw_path)

print("Paths initialized successfully")
print(f"Raw: {raw_path}")

Catalog 'practice_db_catalog' ready.
Schema 'clickstream' created inside 'practice_db_catalog'.
Volume 'data_volume' created inside 'practice_db_catalog.clickstream'.
Paths initialized successfully
Raw: /Volumes/practice_db_catalog/clickstream/data_volume/raw


In [0]:
# Generate reproducible clickstream dataset 
import json, random, datetime
from pathlib import Path

random.seed(42)

# Configuration 

num_users = 60
users = [f"U{u:03d}" for u in range(1, num_users + 1)]
event_types = ["view","click","scroll","add_to_cast","purchase"]
num_events = 700

# Destination 
volume_path = "/Volumes/practice_db_catalog/clickstream/data_volume/clickstream/raw"
output_path = f"{volume_path}/events.json"

# ensure directory exists
Path(volume_path).mkdir(parents=True, exist_ok=True)

# Data Generation Logic 
current = datetime.datetime(2025,1,1,8,0,0)
records = []

for _ in range(num_events):
    user = random.choice(users)
    gap_mins = random.randint(1,120)
    current += datetime.timedelta(minutes=gap_mins)
    records.append({
        "user_id": user,
        "event_time": current.strftime("%Y-%m-%dT%H:%M:%S"),
        "event_type": random.choice(event_types)
    })

# Write to volume 
with open(output_path, "w") as f:
    for r in records:
        json.dump(r, f)
        f.write("\n")

print(f"Generated {len(records)} events for {len(users)} users  {output_path}")

Generated 700 events for 60 users  /Volumes/practice_db_catalog/clickstream/data_volume/clickstream/raw/events.json


%md

# ❓ Scenario Question: Clickstream — Sessionize Events (PySpark) [Easy]




## 🗂️ Scenario

You are working with **raw website clickstream data** captured from multiple user sessions.  
Each event contains a `user_id`, a timestamp (`event_time`), and an `event_type`.  
Your task is to **sessionize** these events — grouping them into sessions based on user activity.

A **session** is defined as a sequence of consecutive user events where the **time gap between events is ≤ 30 minutes** for the same user.  
If there’s a gap of **> 30 minutes** between two events for the same user, a **new session** starts.

---

## 🎯 Task

Perform the following transformations using PySpark:

1. **Read** the input data from `/Volumes/practice_db_catalog/clickstream/data_volume/clickstream/raw/events.json`.  
2. **Order** events for each user by timestamp.  
3. **Identify** session boundaries using a **30-minute inactivity rule**.  
4. **Generate** a unique `session_id` for each user session.  
5. **Calculate** the `session_start` and `session_end` timestamps.  
6. **Display** the resulting DataFrame (`df_final`) with all sessionized events.

---

## 🧩 Assumptions

- Input file is line-delimited JSON at `/Volumes/practice_db_catalog/clickstream/data_volume/clickstream/raw/events.json`.  
- Timestamps are ISO 8601 strings (e.g., `2025-01-02T10:15:30`).  
- A user can have multiple sessions per day.  
- `session_id` can be created by combining `user_id` with an incremental session counter per user.

---

## 📦 Deliverables

- **Output Format:** Spark DataFrame (display via `.show()`)  
- **Expected Columns:**  
  `user_id`, `event_time`, `event_type`, `session_id`, `session_start`, `session_end`

---

## 🧠 Notes

- Use window functions: `lag`, `sum` (cumulative), `first`/`min`, `last`/`max`.  
- Time gap in minutes can be computed via `unix_timestamp` differences.  
- Keep transformations readable and well-commented.

---

## 🧩 Example Output (simplified)

| user_id | event_time          | event_type | session_id | session_start       | session_end         |
|---------|---------------------|------------|------------|---------------------|---------------------|
| U001    | 2025-01-01T10:00:00 | view       | U001_S1    | 2025-01-01T10:00:00 | 2025-01-01T10:20:00 |
| U001    | 2025-01-01T10:20:00 | click      | U001_S1    | 2025-01-01T10:00:00 | 2025-01-01T10:20:00 |
| U001    | 2025-01-01T11:10:00 | purchase   | U001_S2    | 2025-01-01T11:10:00 | 2025-01-01T11:10:00 |

---


In [0]:
df = spark.read.format("json") \
    .load("/Volumes/practice_db_catalog/clickstream/data_volume/clickstream/raw/events.json")

In [0]:
display(df.limit(5))

event_time,event_type,user_id
2025-01-01T08:15:00,view,U041
2025-01-01T08:51:00,click,U048
2025-01-01T09:09:00,view,U015
2025-01-01T10:44:00,purchase,U044
2025-01-01T12:00:00,add_to_cast,U006


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Convert the eventtime as timestamp data type 

df = df.withColumn("event_time",F.to_timestamp("event_time"))

In [0]:
## Order events per user by timestamp 
user_window = Window.partitionBy("user_id").orderBy("event_time")

In [0]:
# identify 30-minute session gaps using lag()

df_gap = df.withColumn(
    "prev_event_time", F.lag("event_time").over(user_window)
).withColumn(
    "gap_minutes",
    (F.unix_timestamp("event_time") - F.unix_timestamp("prev_event_time")) / 60
)

# A new session starts when there's no previous event, or gap > 30 min
df_flag = df_gap.withColumn(
    "is_new_session",
    F.when((F.col("prev_event_time").isNull()) | (F.col("gap_minutes") > 30), 1).otherwise(0)
)

In [0]:
display(df_flag.limit(5))

event_time,event_type,user_id,prev_event_time,gap_minutes,is_new_session
2025-01-05T07:46:00.000Z,view,U001,null,null,1
2025-01-07T05:50:00.000Z,purchase,U001,2025-01-05T07:46:00.000Z,2764.0,1
2025-01-07T12:50:00.000Z,purchase,U001,2025-01-07T05:50:00.000Z,420.0,1
2025-01-08T03:31:00.000Z,click,U001,2025-01-07T12:50:00.000Z,881.0,1
2025-01-08T06:12:00.000Z,view,U001,2025-01-08T03:31:00.000Z,161.0,1


In [0]:
# Session_id

df_session_num = df_flag.withColumn(
    "session_num",
    F.sum("is_new_session").over(user_window.rowsBetween(Window.unboundedPreceding, Window.currentRow))
).withColumn(
    "session_id",
    F.concat_ws("_S", F.col("user_id"), F.col("session_num").cast("string"))
)

In [0]:
df_session_num.limit(5).show()

+-------------------+----------+-------+-------------------+-----------+--------------+-----------+----------+
|         event_time|event_type|user_id|    prev_event_time|gap_minutes|is_new_session|session_num|session_id|
+-------------------+----------+-------+-------------------+-----------+--------------+-----------+----------+
|2025-01-05 07:46:00|      view|   U001|               NULL|       NULL|             1|          1|   U001_S1|
|2025-01-07 05:50:00|  purchase|   U001|2025-01-05 07:46:00|     2764.0|             1|          2|   U001_S2|
|2025-01-07 12:50:00|  purchase|   U001|2025-01-07 05:50:00|      420.0|             1|          3|   U001_S3|
|2025-01-08 03:31:00|     click|   U001|2025-01-07 12:50:00|      881.0|             1|          4|   U001_S4|
|2025-01-08 06:12:00|      view|   U001|2025-01-08 03:31:00|      161.0|             1|          5|   U001_S5|
+-------------------+----------+-------+-------------------+-----------+--------------+-----------+----------+



In [0]:
display(df_session_num.limit(5))

event_time,event_type,user_id,prev_event_time,gap_minutes,is_new_session,session_num,session_id
2025-01-05T07:46:00.000Z,view,U001,null,null,1,1,U001_S1
2025-01-07T05:50:00.000Z,purchase,U001,2025-01-05T07:46:00.000Z,2764.0,1,2,U001_S2
2025-01-07T12:50:00.000Z,purchase,U001,2025-01-07T05:50:00.000Z,420.0,1,3,U001_S3
2025-01-08T03:31:00.000Z,click,U001,2025-01-07T12:50:00.000Z,881.0,1,4,U001_S4
2025-01-08T06:12:00.000Z,view,U001,2025-01-08T03:31:00.000Z,161.0,1,5,U001_S5


In [0]:
# Calculate session_start / session_end per session_id 

session_window = Window.partitionBy("user_id", "session_id")

df_final = df_session_num.withColumn(
    "session_start", F.min("event_time").over(session_window)).withColumn("session_end", F.max("event_time").over(session_window)).select(
        "user_id","event_time","event_type","session_id","session_start","session_end").orderBy("user_id", "event_time")
    
display(df_final.limit(10))
    


user_id,event_time,event_type,session_id,session_start,session_end
U001,2025-01-05T07:46:00.000Z,view,U001_S1,2025-01-05T07:46:00.000Z,2025-01-05T07:46:00.000Z
U001,2025-01-07T05:50:00.000Z,purchase,U001_S2,2025-01-07T05:50:00.000Z,2025-01-07T05:50:00.000Z
U001,2025-01-07T12:50:00.000Z,purchase,U001_S3,2025-01-07T12:50:00.000Z,2025-01-07T12:50:00.000Z
U001,2025-01-08T03:31:00.000Z,click,U001_S4,2025-01-08T03:31:00.000Z,2025-01-08T03:31:00.000Z
U001,2025-01-08T06:12:00.000Z,view,U001_S5,2025-01-08T06:12:00.000Z,2025-01-08T06:12:00.000Z
U001,2025-01-14T04:47:00.000Z,add_to_cast,U001_S6,2025-01-14T04:47:00.000Z,2025-01-14T04:47:00.000Z
U001,2025-01-20T08:10:00.000Z,click,U001_S7,2025-01-20T08:10:00.000Z,2025-01-20T08:10:00.000Z
U001,2025-01-22T22:29:00.000Z,purchase,U001_S8,2025-01-22T22:29:00.000Z,2025-01-22T22:29:00.000Z
U001,2025-01-24T08:43:00.000Z,click,U001_S9,2025-01-24T08:43:00.000Z,2025-01-24T08:43:00.000Z
U001,2025-01-26T18:40:00.000Z,click,U001_S10,2025-01-26T18:40:00.000Z,2025-01-26T18:40:00.000Z


## 🔍 Validation Questions

After creating the final DataFrame (`df_final`), answer these to check your understanding:

1. How many **sessions** does user **U010** have in total?  
2. What is the **average session duration (in minutes)** across all sessions?  
3. Which user has the **maximum number of sessions**? (return user_id and count)  
4. For user **U025**, what is the **start time** of their **first session**?  
5. How many **unique session IDs** were created in total?

In [0]:
# how many sessions does user U010 have in total?

cnt_U010 = df_final.select("session_id").where(df_final.user_id == "U010").distinct().count()
print(f"Number of sessions for user U010: {cnt_U010}")

Number of sessions for user U010: 9


In [0]:
# What is the average session duration (in minutes) across all sessions?

session_durations = (
    df_final
    .select("user_id", "session_id", "session_start", "session_end")
    .dropDuplicates(["user_id", "session_id"])
    .withColumn(
        "session_duration_minutes",
        (F.col("session_end").cast("long") - F.col("session_start").cast("long")) / 60
    )
)

avg_session_duration = session_durations.agg(
    F.round(F.avg("session_duration_minutes"), 2).alias("avg_session_duration_minutes")
)

avg_session_duration.show()

+----------------------------+
|avg_session_duration_minutes|
+----------------------------+
|                        0.15|
+----------------------------+



In [0]:
# Which user has the maximum number of sessions? (return user_id and count)
session_counts = (
    df_final.select("user_id", "session_id")
    .distinct()
    .groupBy("user_id")
    .count()
    .orderBy(F.desc("count"))
)

display(session_counts.limit(1))

user_id,count
U056,18


In [0]:
# For user U025, what is the start time of their first session?

first_session_start = (
    df_final
    .filter(F.col("user_id") == "U025")
    .select("session_id", "session_start")
    .dropDuplicates(["session_id"])
    .orderBy("session_start")
    .limit(1)
)

display(first_session_start)

session_id,session_start
U025_S1,2025-01-02T09:05:00.000Z


In [0]:
# How many unique session IDs were created in total?
unique_sessions_count = df_final.select("session_id").distinct().count()
print(f"Total unique session IDs: {unique_sessions_count}")

Total unique session IDs: 695
